## LIBRARIES

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f, types
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path
import geopandas as gpd
from shapely import wkt
import json
from aux_functions_wealth import *

In [ ]:
projectRoot = Path.cwd().parents[2]

CLEAN_DATA_YELLOW = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "yellow"
CLEAN_DATA_FHVHV = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
GEO_DATA = projectRoot / "data" / "NYC_Taxi_Zones.csv"
# csv files of plot data
TABLES_DIR = projectRoot / "data" / "plotTables" / "economy" / "zones"
HTML_DIR = projectRoot / "data" / "html" / "economy" / "zones"
OUTPUT_DIR = projectRoot / "data" / "clean" / "wealthAnalysis" 
OUTPUT_PATH_Y  = projectRoot / "data" / "clean" / "wealthAnalysis" / "yellow_zone_demand.parquet"
OUTPUT_PATH_FHV  = projectRoot / "data" / "clean" / "wealthAnalysis" / "fhvhv_zone_demand.parquet"

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## SPARK CONFIGURATION

In [ ]:
findspark.init()

#Adjust these settings based on your system's resources and requirements


# spark = (SparkSession.builder
#         .master("local[6]")  
#         .config("spark.driver.memory", "8g")
#         .config("spark.executor.memory", "8g")
#         .config("spark.sql.shuffle.partitions", "100")
#         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
#         .config("spark.driver.maxResultSize", "1g")
#         .config("spark.sql.session.timeZone", "America/New_York")
#         .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
#         # Network binding fixes for macOS
#         .config("spark.driver.host", "127.0.0.1")
#         .config("spark.driver.bindAddress", "127.0.0.1")
#         .getOrCreate())

spark = (SparkSession.builder
        .master("local[*]")  
        .config("spark.driver.memory", "16g")  
        .config("spark.executor.memory", "16g") 
        .config("spark.memory.fraction", "0.8") 
        .config("spark.sql.shuffle.partitions", "200")
        .config("spark.sql.adaptive.enabled", "true") 
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m") 
        .config("spark.driver.maxResultSize", "4g") 
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

## LOAD DATASETS

In [ ]:
# YELLOW
yellow = spark.read.parquet(str(CLEAN_DATA_YELLOW))
print(yellow.columns)
yellow = yellow.select(
        f.col("PULocationID").cast(types.IntegerType()),
        f.col("DOLocationID").cast(types.IntegerType()),
        f.col("pickup_datetime"),
        f.col("dropoff_datetime"),
        f.col("trip_distance").alias("trip_miles"),
        f.col("trip_time").cast("long").alias("trip_time"),
        f.col("total_amount"),
        f.col("tip_amount").alias("tips"),
        f.col("pickup_borough"),
        f.col("pickup_zone"),
        f.col("dropoff_borough"),
        f.col("dropoff_zone")
    )

# FHVHV
fhv = spark.read.parquet(str(CLEAN_DATA_FHVHV))
print(fhv.columns)
fhv = fhv.select(
        f.col("PULocationID").cast(types.IntegerType()),
        f.col("DOLocationID").cast(types.IntegerType()),
        f.col("pickup_datetime"),
        f.col("dropoff_datetime"),
        f.col("trip_miles"),
        f.col("trip_time").cast("long").alias("trip_time"),
        f.col("total_amount"),
        f.col("tips"),
        f.col("pickup_borough"),
        f.col("pickup_zone"),
        f.col("dropoff_borough"),
        f.col("dropoff_zone")
    )

# ZONE
df_geodata = pd.read_csv(GEO_DATA)
df_geodata = df_geodata.rename(columns ={"the_geom" : "geometry", "shape_leng" : "length", "shape_area" : "area", "locationid" : "id"})
# Convierte WKT → geometría
df_geodata['geometry'] = df_geodata['geometry'].apply(wkt.loads) 
gdf = gpd.GeoDataFrame(df_geodata, crs='EPSG:4326')
gdf_proj = gdf.to_crs('EPSG:2263')
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['lat'] = gdf_proj.centroid.to_crs('EPSG:4326').y
gdf['lon'] = gdf_proj.centroid.to_crs('EPSG:4326').x

In [ ]:
yellow, fhv = count_trips(yellow, fhv)
yellow, fhv = time_band_calc(yellow, fhv)
yellow, fhv = tip_pct_calc(yellow, fhv)
n_days_yellow, n_days_fhv = range_days(yellow, fhv)
agg_yellow, agg_fhv = aggregations_yellow_fhv(yellow, fhv, n_days_yellow, n_days_fhv)
result_yellow, result_fhv = pickup_time_band_results(yellow, fhv, agg_yellow, agg_fhv, n_days_yellow, n_days_fhv)

In [ ]:
demand_yellow = result_yellow.toPandas()
demand_fhv = result_fhv.toPandas()
demand_yellow.to_parquet(OUTPUT_PATH_Y, compression='snappy')
demand_fhv.to_parquet(OUTPUT_PATH_FHV, compression='snappy')